# C_vote_t: v20의 한글표만 72B 표로 바꾼 v20t + HanSunwook + 112 3자 투표
**GPU 런타임(L4)**으로 위에서부터 실행하세요. 마지막에 `C_vote_t: PASS`가 나오면 `out/C_vote_t.zip`이 생깁니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys
D = "/content/drive/MyDrive/Projects/항만문제해결을위한AI기술공모전/combo_kit"
os.chdir(D)
!ls -la make_v20t.py catalog_ko_72b.json v20.zip 112_submit.zip 1031_08_hyb_w.zip build_vote.py
!pip -q install lightgbm==4.6.0 catboost==1.2.8 rapidfuzz==3.14.3 transformers==4.57.6 accelerate==1.9.0 scikit-learn==1.5.2
W = "/content/combo"; os.makedirs(W, exist_ok=True); os.makedirs(f"{D}/out", exist_ok=True)
for f in ("v20t.zip", "C_vote_t.zip"):
    if os.path.exists(f"{W}/{f}"): os.remove(f"{W}/{f}")


In [ ]:
# v20t 만들기 (표 파일만 교체) → 3자 투표 조립 (약 20분)
!python make_v20t.py {W}/v20t.zip v20.zip catalog_ko_72b.json
!unzip -p {W}/v20t.zip model/catalog_ko.json | head -c 200; echo
!python build_vote.py {W}/C_vote_t.zip {W}/v20t.zip 1031_08_hyb_w.zip 112_submit.zip 4 ko
os.remove(f"{W}/v20t.zip")


In [ ]:
# 점검 (약 20분): 3,000행 + 영문 불변 + 600행 행 독립성. 통과할 때만 out/으로 복사
zp = f"{W}/C_vote_t.zip"
c = subprocess.run([sys.executable, "check_combo.py", zp, f"{W}/work_t"], capture_output=True, text=True)
log = c.stdout + "\n" + c.stderr
open(f"{W}/C_vote_t_check.log", "w").write(log)
print("\n".join(l for l in log.splitlines() if l.startswith(("[1]", "[2]", "[3]", "zip size", "OK", "PROBLEM", "WARNING ko", "WARNING hs")) or "predictions to output" in l))
bad = [k for k in ("WARNING ko", "WARNING hs") if k in log]
last = (c.stdout.strip().splitlines() or [""])[-1]
if c.returncode == 0 and last == "OK" and not bad:
    shutil.copy(zp, f"{D}/out/C_vote_t.zip"); print("C_vote_t: PASS ->", round(os.path.getsize(f"{D}/out/C_vote_t.zip")/1e9, 2), "GB")
else:
    print("C_vote_t: FAIL - 올리지 마세요.", bad or last)
